In [ ]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 109.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.5 MB/s eta 0:00:00


In [ ]:
# Run in Colab with T4 GPU selected (Runtime -> Change runtime type -> T4 GPU)
# !pip install -q transformers trl accelerate peft bitsandbytes datasets

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = "train-2.jsonl"
OUTPUT_DIR = "./qwen2.5-1.5b-medvqa-lora"

# ---- 4-bit quantization (fits comfortably on T4's 16GB) ----
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,   # T4 = Turing, no bf16 support
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,   # keep everything fp16, consistent with T4
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

dataset = load_dataset("json", data_files={"train": TRAIN_FILE})
print(f"Training examples: {len(dataset['train'])}")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,   # AMP off -- avoids GradScaler crashing on stray bf16 LoRA tensors
    bf16=False,   # T4 doesn't support bf16 at all
    max_length=512,
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    peft_config=lora_config,
    processing_class=tokenizer,
)

print("Starting training...")
trainer.train()

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Done! Adapter saved to '{OUTPUT_DIR}'")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

... (9 outputs left out)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Starting training...


Step,Training Loss
10,2.747969
20,0.664761
30,0.400901
40,0.372788
50,0.342197
60,0.328848
70,0.328395
80,0.321154
90,0.303963
100,0.298003


Done! Adapter saved to './qwen2.5-1.5b-medvqa-lora'


##TESTING

In [ ]:
!pip install -q nltk rouge_score

  Preparing metadata (setup.py) ... done


In [ ]:
# Run in Colab with T4 GPU selected
# !pip install -q transformers peft accelerate bitsandbytes datasets pandas openpyxl nltk rouge_score

import re
import string
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "./qwen2.5-1.5b-medvqa-lora"
TEST_FILE = "test.xlsx"
OUTPUT_FILE = "eval_results.xlsx"
MAX_NEW_TOKENS = 32

SYSTEM_PROMPT = (
    "You are a medical imaging assistant. You will be given a question about "
    "a radiology image and a list of clinically relevant entities that were "
    "automatically extracted from the image's findings using an NER model. "
    "The entity list may occasionally be incomplete or contain minor errors, "
    "so use it as supporting context rather than as a fully certain ground truth. "
    "If no entities are available, answer using your own medical knowledge and "
    "the question alone.\n\n"
    "Answer with a single word or short phrase only, matching the concise style "
    "shown in training (e.g. 'yes', 'no', 'cardiovascular'). Do not explain your "
    "reasoning or add extra commentary."
)

def build_user_prompt(question, entities):
    entities = entities if isinstance(entities, str) and entities.strip() else "None available"
    return f"Question: {question}\nExtracted entities: {entities}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model + adapter...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

def generate_answer(question, entities):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question, entities)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\s+", " ", s).strip()
    return s

smoothing = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def token_f1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return int(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)
    return 2 * precision * recall / (precision + recall)

def bleu1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    return sentence_bleu([gold_tokens], pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothing)

def rouge_scores(pred, gold):
    scores = rouge.score(normalize_text(gold), normalize_text(pred))
    return scores["rouge1"].fmeasure, scores["rougeL"].fmeasure

def is_closed_question(gold_answer):
    return normalize_text(gold_answer) in ("yes", "no")

print("Loading test set...")
test_df = pd.read_excel(TEST_FILE).reset_index(drop=True)
print(f"Evaluating on {len(test_df)} held-out examples (with real extracted entities)...")

results = []
for i, row in test_df.iterrows():
    pred = generate_answer(row["question"], row["Classified_Keywords"])
    gold = str(row["answer"])

    r1, rL = rouge_scores(pred, gold)

    results.append({
        "question": row["question"],
        "entities": row["Classified_Keywords"],
        "gold_answer": gold,
        "predicted_answer": pred,
        "is_closed": is_closed_question(gold),
        "exact_match": exact_match(pred, gold),
        "token_f1": token_f1(pred, gold),
        "bleu1": bleu1(pred, gold),
        "rouge1": r1,
        "rougeL": rL,
    })

    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(test_df)} done...")

results_df = pd.DataFrame(results)
results_df.to_excel(OUTPUT_FILE, index=False)

closed_df = results_df[results_df["is_closed"]]
open_df = results_df[~results_df["is_closed"]]

print()
print("=" * 55)
print(f"Overall exact match accuracy: {results_df['exact_match'].mean():.4f}")
print(f"Overall token F1:             {results_df['token_f1'].mean():.4f}")
print()
print(f"Closed-set (yes/no) accuracy: {closed_df['exact_match'].mean():.4f}  (n={len(closed_df)})")
print()
print("--- Open-set (free-text) metrics -- exact match is a strict lower bound here ---")
print(f"Open-set exact match:  {open_df['exact_match'].mean():.4f}  (n={len(open_df)})")
print(f"Open-set token F1:     {open_df['token_f1'].mean():.4f}")
print(f"Open-set BLEU-1:       {open_df['bleu1'].mean():.4f}")
print(f"Open-set ROUGE-1:      {open_df['rouge1'].mean():.4f}")
print(f"Open-set ROUGE-L:      {open_df['rougeL'].mean():.4f}")
print("=" * 55)
print(f"\nFull results saved to '{OUTPUT_FILE}'")

Loading base model + adapter...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading test set...
Evaluating on 269 held-out examples (with real extracted entities)...
  25/269 done...
  50/269 done...
  75/269 done...
  100/269 done...
  125/269 done...
  150/269 done...
  175/269 done...
  200/269 done...
  225/269 done...
  250/269 done...

Overall exact match accuracy: 0.5019
Overall token F1:             0.5673

Closed-set (yes/no) accuracy: 0.6644  (n=149)

--- Open-set (free-text) metrics -- exact match is a strict lower bound here ---
Open-set exact match:  0.3000  (n=120)
Open-set token F1:     0.4466
Open-set BLEU-1:       0.3949
Open-set ROUGE-1:      0.4466
Open-set ROUGE-L:      0.4466

Full results saved to 'eval_results.xlsx'


In [ ]:
# Run in Colab with T4 GPU selected
# !pip install -q nltk rouge_score

import re
import string
import torch
import pandas as pd
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "./qwen2.5-1.5b-medvqa-lora"
TEST_FILE = "test.xlsx"
OUTPUT_FILE = "eval_results.xlsx"
MAX_NEW_TOKENS = 32

SYSTEM_PROMPT = (
    "You are a medical imaging assistant. You will be given a question about "
    "a radiology image and a list of clinically relevant entities that were "
    "automatically extracted from the image's findings using an NER model. "
    "The entity list may occasionally be incomplete or contain minor errors, "
    "so use it as supporting context rather than as a fully certain ground truth. "
    "If no entities are available, answer using your own medical knowledge and "
    "the question alone.\n\n"
    "Answer with a single word or short phrase only, matching the concise style "
    "shown in training (e.g. 'yes', 'no', 'cardiovascular'). Do not explain your "
    "reasoning or add extra commentary."
)

def build_user_prompt(question, entities):
    entities = entities if isinstance(entities, str) and entities.strip() else "None available"
    return f"Question: {question}\nExtracted entities: {entities}"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model + adapter...")
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
)
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

def generate_answer(question, entities):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(question, entities)},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def normalize_text(s):
    s = str(s).lower()
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    s = "".join(ch for ch in s if ch not in string.punctuation)
    s = re.sub(r"\s+", " ", s).strip()
    return s

smoothing = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

def exact_match(pred, gold):
    return int(normalize_text(pred) == normalize_text(gold))

def token_recall(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not gold_tokens:
        return int(pred_tokens == gold_tokens)
    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())
    return num_same / len(gold_tokens)

def bleu1(pred, gold):
    pred_tokens = normalize_text(pred).split()
    gold_tokens = normalize_text(gold).split()
    if not pred_tokens or not gold_tokens:
        return 0.0
    return sentence_bleu([gold_tokens], pred_tokens, weights=(1, 0, 0, 0), smoothing_function=smoothing)

def rouge_scores(pred, gold):
    scores = rouge.score(normalize_text(gold), normalize_text(pred))
    return scores["rouge1"].fmeasure, scores["rougeL"].fmeasure

def is_closed_question(gold_answer):
    return normalize_text(gold_answer) in ("yes", "no")

print("Loading test set...")
test_df = pd.read_excel(TEST_FILE).reset_index(drop=True)
print(f"Evaluating on {len(test_df)} held-out examples (with real extracted entities)...")

results = []
for i, row in test_df.iterrows():
    pred = generate_answer(row["question"], row["Classified_Keywords"])
    gold = str(row["answer"])

    r1, rL = rouge_scores(pred, gold)

    results.append({
        "question": row["question"],
        "entities": row["Classified_Keywords"],
        "gold_answer": gold,
        "predicted_answer": pred,
        "is_closed": is_closed_question(gold),
        "exact_match": exact_match(pred, gold),
        "token_recall": token_recall(pred, gold),
        "bleu1": bleu1(pred, gold),
        "rouge1": r1,
        "rougeL": rL,
    })

    if (i + 1) % 25 == 0:
        print(f"  {i + 1}/{len(test_df)} done...")

results_df = pd.DataFrame(results)
results_df.to_excel(OUTPUT_FILE, index=False)

closed_df = results_df[results_df["is_closed"]]
open_df = results_df[~results_df["is_closed"]]

print()
print("=" * 55)
print(f"Overall exact match accuracy: {results_df['exact_match'].mean():.4f}")
print()
print(f"Closed-set (yes/no) accuracy: {closed_df['exact_match'].mean():.4f}  (n={len(closed_df)})")
print()
print("--- Open-set (free-text) metrics -- exact match is a strict lower bound here ---")
print(f"Open-set exact match:  {open_df['exact_match'].mean():.4f}  (n={len(open_df)})")
print(f"Open-set token recall: {open_df['token_recall'].mean():.4f}")
print(f"Open-set BLEU-1:       {open_df['bleu1'].mean():.4f}")
print(f"Open-set ROUGE-1:      {open_df['rouge1'].mean():.4f}")
print(f"Open-set ROUGE-L:      {open_df['rougeL'].mean():.4f}")
print("=" * 55)
print(f"\nFull results saved to '{OUTPUT_FILE}'")

Loading base model + adapter...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading test set...
Evaluating on 269 held-out examples (with real extracted entities)...
  25/269 done...
  50/269 done...
  75/269 done...
  100/269 done...
  125/269 done...
  150/269 done...
  175/269 done...
  200/269 done...
  225/269 done...
  250/269 done...

Overall exact match accuracy: 0.5019

Closed-set (yes/no) accuracy: 0.6644  (n=149)

--- Open-set (free-text) metrics -- exact match is a strict lower bound here ---
Open-set exact match:  0.3000  (n=120)
Open-set token recall: 0.4545
Open-set BLEU-1:       0.3949
Open-set ROUGE-1:      0.4466
Open-set ROUGE-L:      0.4466

Full results saved to 'eval_results.xlsx'
